# Notebook 3 — Atomic Structure & Defect Modelling

## Goal

Learn how pristine crystal structures are transformed into defect-containing models for atomistic calculations.

The core idea is:

**bulk structure → supercell → introduce defect → generate candidate defect structures → compare relaxed configurations**

## 1. Build a Supercell

- start from a bulk structure
- get primitive structure
- create a larger periodic cell
- inspect the number of atoms and lattice dimensions
- understand why defects are usually calculated in supercells

## 2. Generate Point Defects and charge states

- vacancy
- substitution
- interstitial
- identify the defect site
- inspect the resulting structure
- assign possible charge states

## 3. Structural Distortions

- generate alternative local distortions
- understand why different starting structures may relax to different minima

## 4. Compare Relaxed Defect Structures

- compare final energies
- compare local atomic environments
- identify the lowest-energy defect geometry

## 1. Build a Supercell

In [1]:
from pymatgen.io.vasp.inputs import Potcar, Kpoints, Incar, Poscar, VaspInput
from monty.serialization import loadfn
from pathlib import Path
from pymatgen.core import Structure
import doped

In [ ]:
# Start from a bulk structure
# Get entry from Material Project
from doped.chemical_potentials import get_entries
mgo_entries = get_entries("MgO", api_key="UyZUs9gIm3i9StsZRk5CAPQVL9dttHEY")

print("Object type:", type(mgo_entries))
print("Number of entries:", len(mgo_entries))
first_entry = mgo_entries[0]

#test
print("Entry type:", type(first_entry))
print(first_entry)


Object type: <class 'list'>
Number of entries: 23
Entry type: <class 'pymatgen.core.entries.ComputedStructureEntry'>
{'identifier': 'mp-aaaaabwr', 'suffix': 'GGA', 'separator': '-'} ComputedStructureEntry - Mg1 O1       (MgO)
Energy (Uncorrected)     = -11.9680  eV (-5.9840  eV/atom)
Correction               = -0.6870   eV (-0.3435  eV/atom)
Energy (Final)           = -12.6550  eV (-6.3275  eV/atom)
Energy Adjustments:
  MP2020 anion correction (oxide): -0.6870   eV (-0.3435  eV/atom)
Parameters:
  potcar_spec            = [PotcarSpec(titel='PAW_PBE Mg_pv 06Sep2000', hash='bbcf6f81cc34a3090d483ad641178746', summary_stats=None), PotcarSpec(titel='PAW_PBE O 08Apr2002', hash='7a25bc5b9a5393f46600a4939d357982', summary_stats=None)]
  run_type               = GGA
Data:
  oxide_type             = oxide
  aspherical             = True
  last_updated           = 2026-05-29 05:02:50.065987+00:00
  task_id                = aaafauzb
  material_id            = mp-aaaaabwr
  oxidation_states       

In [6]:
# Check the first entry's attributes
print("Entry ID:", first_entry.entry_id)
print("Energy per atom:", first_entry.energy_per_atom)
print("Composition:", first_entry.composition)
print("Structure:", first_entry.structure)

Entry ID: {'identifier': 'mp-aaaaabwr', 'suffix': 'GGA', 'separator': '-'}
Energy per atom: -6.32752468
Composition: Mg1 O1
Structure: Full Formula (Mg1 O1)
Reduced Formula: MgO
abc   :   3.009789   3.009789   3.009789
angles:  60.000000  60.000000  60.000000
pbc   :       True       True       True
Sites (2)
  #  SP      a    b    c    magmom
---  ----  ---  ---  ---  --------
  0  Mg    0    0    0          -0
  1  O     0.5  0.5  0.5        -0


In [7]:
# Select the entry with the lowest energy per atom
mgo_entry = sorted(mgo_entries, key=lambda x: x.energy_per_atom,reverse = False)[0]

In [8]:
# Get primitive structure
from doped.utils.symmetry import get_primitive_structure
prim_struc = get_primitive_structure(mgo_entry.structure)  # get a clean primitive structure
print("Primitive structure:", prim_struc)

Primitive structure: Full Formula (Mg1 O1)
Reduced Formula: MgO
abc   :   3.009789   3.009789   3.009789
angles:  60.000000  60.000000  60.000000
pbc   :       True       True       True
Sites (2)
  #  SP      a    b    c    magmom
---  ----  ---  ---  ---  --------
  0  Mg    0    0    0          -0
  1  O     0.5  0.5  0.5        -0


In [ ]:
# Compare the original and primitive structures
print("Original composition:", mgo_entry.structure.composition)
print("Primitive composition:", prim_struc.composition)

print("Original number of sites:", mgo_entry.structure.num_sites)
print("Primitive number of sites:", prim_struc.num_sites)

print("Original lattice lengths:", mgo_entry.structure.lattice.abc)
print("Primitive lattice lengths:", prim_struc.lattice.abc)

print("Original lattice angles:", mgo_entry.structure.lattice.angles)
print("Primitive lattice angles:", prim_struc.lattice.angles)

## Coincidently the Original structure is a primitive structure3

Original composition: Mg1 O1
Primitive composition: Mg1 O1
Original number of sites: 2
Primitive number of sites: 2
Original lattice lengths: (3.0097887004120407, 3.0097887004120407, 3.0097887004120407)
Primitive lattice lengths: (3.0097887004120407, 3.0097887004120407, 3.0097887004120407)
Original lattice angles: (59.99999999999999, 59.99999999999999, 59.99999999999999)
Primitive lattice angles: (59.99999999999999, 59.99999999999999, 59.99999999999999)


In [11]:
# Make a copy so the primitive structure itself is not modified
supercell_manual = prim_struc.copy()

# Build a 2 × 2 × 2 supercell
supercell_manual.make_supercell([2, 2, 2])

print("2 × 2 × 2 supercell")
print("Number of sites:", supercell_manual.num_sites)
print("Lattice lengths:", supercell_manual.lattice.abc)
print("Volume:", supercell_manual.volume)

assert supercell_manual.num_sites == prim_struc.num_sites * 8

volume_ratio = supercell_manual.volume / prim_struc.volume

print("Atom-number ratio:", supercell_manual.num_sites / prim_struc.num_sites)
print("Volume ratio:", volume_ratio)

2 × 2 × 2 supercell
Number of sites: 16
Lattice lengths: (6.019577400824081, 6.019577400824081, 6.019577400824081)
Volume: 154.235026122732
Atom-number ratio: 8.0
Volume ratio: 8.0


### Why use a supercell?

Under periodic boundary conditions, a defect is repeated throughout the crystal.

A larger supercell increases the distance between periodic defect images and reduces artificial defect–defect interactions.

However, larger supercells also require more computational resources.

Therefore, supercell construction is a balance between:

**reducing finite-size effects ↔ controlling computational cost**

## Automatic Supercell Generation with `doped`

In a practical defect workflow, the supercell does not always need to be chosen manually.

`doped.DefectsGenerator` can automatically search for a suitable supercell based on geometric and computational constraints.

In [ ]:
from doped.generation import DefectsGenerator

defect_gen = DefectsGenerator(
    structure=prim_struc,
    supercell_gen_kwargs={"force_cubic": True}
)

print("Manual supercell:")
print("Sites:", supercell_manual.num_sites)
print("Lattice:", supercell_manual.lattice.abc)

auto_supercell = defect_gen.bulk_supercell
supercell_matrix = defect_gen.supercell_matrix

print("Automatic supercell matrix:")
print(supercell_matrix)

#Compare the two different supercell

print("\nAutomatic bulk supercell:")
print("Number of sites:", auto_supercell.num_sites)
print("Lattice lengths:", auto_supercell.lattice.abc)
print("Volume:", auto_supercell.volume)

print("\nAutomatic doped supercell:")
print("Sites:", auto_supercell.num_sites)
print("Lattice:", auto_supercell.lattice.abc)

Guessing charge states: 100.0%|██████████| [00:00,  278.09it/s]                                

Vacancies    Guessed Charges    Conv. Cell Coords    Wyckoff
-----------  -----------------  -------------------  ---------
v_Mg         [+1,0,-1,-2]       [0.000,0.000,0.000]  4a
v_O          [+2,+1,0,-1]       [0.500,0.500,0.500]  4b

Substitutions    Guessed Charges    Conv. Cell Coords    Wyckoff
---------------  -----------------  -------------------  ---------
Mg_O             [+4,+3,+2,+1,0]    [0.500,0.500,0.500]  4b
O_Mg             [0,-1,-2,-3,-4]    [0.000,0.000,0.000]  4a

Interstitials    Guessed Charges    Conv. Cell Coords    Wyckoff
---------------  -----------------  -------------------  ---------
Mg_i_Td          [+2,+1,0]          [0.250,0.250,0.250]  8c
O_i_Td           [0,-1,-2]          [0.250,0.250,0.250]  8c

The number in the Wyckoff label is the site multiplicity/degeneracy of that defect in the conventional ('conv.') unit cell, which comprises 4 formula unit(s) of MgO.

Manual supercell:
Sites: 16
Lattice: (6.019577400824081, 6.019577400824081, 6.019577400824

## Manual vs Automatic Supercell Generation

The manually generated `2 × 2 × 2` supercell is useful for understanding how a crystal is expanded.

In a real defect workflow, however, the most suitable supercell may not be a simple integer multiple such as `2 × 2 × 2`.

`doped` searches for a supercell that satisfies practical constraints such as defect-image separation and computational cost.

The key idea is:

**manual construction helps understanding; automated generation helps practical calculations.**

## 2. Generate Point Defects

Once a suitable supercell is available, point defects can be generated automatically.

For an intrinsic material, the main point-defect types are:

- vacancy: remove an atom
- substitution: replace one atom with another host element
- interstitial: add an atom at an empty site

In [ ]:
## observate api
print(defect_gen.defects.keys())

dict_keys(['vacancies', 'substitutions', 'interstitials'])


In [ ]:
# further observation
for defect_type, defects in defect_gen.defects.items():
    print(defect_type)
    print(defects)
    print()

vacancies
[v_Mg vacancy defect at site [0.000,0.000,0.000] in structure, v_O vacancy defect at site [0.500,0.500,0.500] in structure]

substitutions
[Mg_O substitution defect at site [0.500,0.500,0.500] in structure, O_Mg substitution defect at site [0.000,0.000,0.000] in structure]

interstitials
[Mg_i interstitial defect at site [0.250,0.250,0.250] in structure, O_i interstitial defect at site [0.250,0.250,0.250] in structure]



### Vacancy

A vacancy is created by removing an atom from one of its lattice sites.

For MgO:

- `v_Mg`: Mg vacancy
- `v_O`: O vacancy

In [16]:
defect_gen.defects["vacancies"]

[v_Mg vacancy defect at site [0.000,0.000,0.000] in structure,
 v_O vacancy defect at site [0.500,0.500,0.500] in structure]

### Substitution

A substitutional defect is created when an atom occupies the lattice site of another species.

For MgO:

- `Mg_O`: Mg occupying an O site
- `O_Mg`: O occupying an Mg site

In [17]:
defect_gen.defects["substitutions"]

[Mg_O substitution defect at site [0.500,0.500,0.500] in structure,
 O_Mg substitution defect at site [0.000,0.000,0.000] in structure]

### Interstitial

An interstitial defect is created by placing an additional atom at a normally unoccupied position in the crystal.

For MgO:

- `Mg_i`
- `O_i`

In [18]:
defect_gen.defects["interstitials"]

[Mg_i interstitial defect at site [0.250,0.250,0.250] in structure,
 O_i interstitial defect at site [0.250,0.250,0.250] in structure]

## Inspect Defect Charge States

The same structural defect can exist in different charge states.

For example, an oxygen vacancy may be considered in several possible charge states because different numbers of electrons can be removed from or added to the defect system.

In [19]:
print(defect_gen.defect_entries.keys())

dict_keys(['v_Mg_+1', 'v_Mg_0', 'v_Mg_-1', 'v_Mg_-2', 'v_O_+2', 'v_O_+1', 'v_O_0', 'v_O_-1', 'Mg_O_+4', 'Mg_O_+3', 'Mg_O_+2', 'Mg_O_+1', 'Mg_O_0', 'O_Mg_0', 'O_Mg_-1', 'O_Mg_-2', 'O_Mg_-3', 'O_Mg_-4', 'Mg_i_Td_+2', 'Mg_i_Td_+1', 'Mg_i_Td_0', 'O_i_Td_0', 'O_i_Td_-1', 'O_i_Td_-2'])


In [21]:
first_name = list(defect_gen.defect_entries.keys())[0]

entry = defect_gen.defect_entries[first_name]

print(first_name)
print(type(entry))

print("Charge state:", entry.charge_state)
print("Defect:", entry.defect)

v_Mg_+1
<class 'doped.core.DefectEntry'>
Charge state: 1
Defect: v_Mg vacancy defect at site [0.000,0.000,0.000] in structure


## Inspect the Defect Supercell

A defect is represented by modifying one site in the bulk supercell.

Therefore, it is useful to compare the pristine supercell and a generated defect structure.

In [23]:
entry = defect_gen.defect_entries[first_name]

bulk_supercell = defect_gen.bulk_supercell
defect_supercell = entry.defect_supercell

print("Bulk supercell atoms:", bulk_supercell.num_sites)
print("Defect supercell atoms:", defect_supercell.num_sites)

Bulk supercell atoms: 216
Defect supercell atoms: 215


## 3。 structural distortions

Generate Distorted Defect Structures

A defect does not necessarily relax to the same atomic geometry from every starting structure.

To search for lower-energy defect configurations, `ShakeNBreak` generates several distorted versions of each defect structure.

The idea is:

**one defect → multiple distorted starting structures → relaxation → compare final energies**

### Why generate distorted structures?

A defect may have several possible local atomic arrangements.

If we start from only one symmetric structure, the relaxation may remain trapped in a local minimum and miss a lower-energy configuration.

Therefore, several distorted starting structures are generated to explore different regions of the potential energy surface.

In [24]:
from shakenbreak.input import Distortions

Dist = Distortions(defect_entries=defect_gen)

Oxidation states were not explicitly set, thus have been guessed as {'Mg': 2, 'O': -2}. If this is unreasonable you should manually set oxidation_states
